# Fase 2/Fase 3 - Pipeline Big Data Gestion Territorial El Monte

Notebook reproducible para ejecutar el pipeline lakehouse medallion en PySpark. La version queda preparada para `SCALE_MODE = "full"` en Google Colab High-RAM, sin reducir filas ni fingir ejecucion completa. Si el entorno no soporta la corrida, el notebook guarda evidencia controlada del fallo y recomienda Dataproc/Spark Serverless.

## 1. Preparacion del entorno Colab/local

En Colab High-RAM usar runtime sin GPU. No escribir Parquet masivo en Google Drive durante la ejecucion; la raiz operativa debe ser `/content/fase2_gestion_territorial_el_monte` y Drive solo se usa al final para copiar artefactos.

In [19]:
# En Colab, descomentar si PySpark no esta instalado.
# !pip install -q pyspark==3.5.1 psutil

import json
import os
import platform
import shutil
import sys
import time
import zipfile
from pathlib import Path

try:
    import psutil
except ImportError:
    psutil = None

print("Python:", platform.python_version())
if psutil:
    print("RAM total GB:", round(psutil.virtual_memory().total / 1024**3, 2))
    print("RAM libre GB:", round(psutil.virtual_memory().available / 1024**3, 2))
else:
    print("psutil no disponible; instalar con pip si se requiere diagnostico de RAM.")

disk_probe = Path('/content') if Path('/content').exists() else Path.cwd()
print("Disco total GB:", round(shutil.disk_usage(disk_probe).total / 1024**3, 2))
print("Disco libre GB:", round(shutil.disk_usage(disk_probe).free / 1024**3, 2))

if psutil and psutil.virtual_memory().available / 1024**3 < 20:
    print("ADVERTENCIA: RAM libre menor a 20 GB. Full puede fallar en Colab/local.")
if shutil.disk_usage(disk_probe).free / 1024**3 < 80:
    print("ADVERTENCIA: disco libre menor a 80 GB. Full puede fallar durante escrituras Parquet.")

Python: 3.12.13
RAM total GB: 12.67
RAM libre GB: 3.62
Disco total GB: 107.72
Disco libre GB: 81.57
ADVERTENCIA: RAM libre menor a 20 GB. Full puede fallar en Colab/local.


## 2. Rutas de trabajo seguras

Estas rutas evitan escribir datos masivos directamente en Google Drive. Si se ejecuta localmente, el notebook usa la raiz del repositorio detectada.

In [11]:
IN_COLAB = Path('/content').exists()

if IN_COLAB:
    PROJECT_ROOT = Path('/content/fase2_gestion_territorial_el_monte')
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / 'data'
EVIDENCE_ROOT = PROJECT_ROOT / 'evidence'
TMP_ROOT = Path('/content/spark_tmp') if IN_COLAB else PROJECT_ROOT / 'work' / 'spark_tmp'
WAREHOUSE_DIR = Path('/content/spark_warehouse') if IN_COLAB else PROJECT_ROOT / 'work' / 'spark_warehouse'
CHECKPOINT_DIR = Path('/content/spark_checkpoints') if IN_COLAB else PROJECT_ROOT / 'work' / 'spark_checkpoints'

for folder in [PROJECT_ROOT, DATA_ROOT, EVIDENCE_ROOT, TMP_ROOT, WAREHOUSE_DIR, CHECKPOINT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Limpieza segura: solo temporales Spark, nunca Drive ni codigo fuente.
for folder in [TMP_ROOT, WAREHOUSE_DIR, CHECKPOINT_DIR]:
    if str(folder).startswith('/content') or str(folder).startswith(str(PROJECT_ROOT / 'work')):
        shutil.rmtree(folder, ignore_errors=True)
        folder.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('EVIDENCE_ROOT:', EVIDENCE_ROOT)
print('TMP_ROOT:', TMP_ROOT)

PROJECT_ROOT: /content/fase2_gestion_territorial_el_monte
DATA_ROOT: /content/fase2_gestion_territorial_el_monte/data
EVIDENCE_ROOT: /content/fase2_gestion_territorial_el_monte/evidence
TMP_ROOT: /content/spark_tmp


## 3. Parametros de escala

La version final queda preparada para `full`: 50.000 solicitudes, 10.800.000 eventos y 1.500.000 logs. Se puede cambiar a `small` o `medium` solo para pruebas rapidas, no para reportar una ejecucion full ficticia.

In [12]:
import sys
print(sys.path)

['/content/fase2_gestion_territorial_el_monte/src', '/content/spark_tmp/spark-25aa6c2c-4a8d-4f33-b727-736e2f5e9cf9/userFiles-362abe74-6131-4697-b400-1b2bbd3225a5', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']


In [13]:
zip_file_path = Path('/content/fase2_colab_full_robusto_sin_data.zip')

if zip_file_path.exists():
    print(f'Unzipping {zip_file_path} to {PROJECT_ROOT}')
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(PROJECT_ROOT)
    print('Unzipping complete.')
else:
    print(f'Error: Zip file not found at {zip_file_path}')

# Verify if the src directory and territorial_pipeline module are now present
expected_src_path = PROJECT_ROOT / 'src'
expected_module_path = PROJECT_ROOT / 'src' / 'territorial_pipeline'

if expected_src_path.exists():
    print(f'src directory found at: {expected_src_path}')
    if expected_module_path.exists():
        print(f'territorial_pipeline module found at: {expected_module_path}')
    else:
        print(f'Warning: territorial_pipeline module not found within {expected_src_path}')
else:
    print(f'Error: src directory not found at: {expected_src_path} after unzipping.')

# Add the src directory to sys.path again, in case it wasn't there before unzipping
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))


Unzipping /content/fase2_colab_full_robusto_sin_data.zip to /content/fase2_gestion_territorial_el_monte
Unzipping complete.
src directory found at: /content/fase2_gestion_territorial_el_monte/src
territorial_pipeline module found at: /content/fase2_gestion_territorial_el_monte/src/territorial_pipeline


In [14]:
from territorial_pipeline.config import get_profile
from territorial_pipeline.pipeline import run_pipeline

SCALE_MODE = os.environ.get('SCALE_MODE', 'full')
DRIVER_MEMORY = os.environ.get('SPARK_DRIVER_MEMORY', '20g')  # Alternativas Colab si falla: 16g o 12g.

profile = get_profile(SCALE_MODE)
expected_bronze_rows = (
    profile.solicitudes
    + profile.eventos
    + profile.logs
    + int(profile.solicitudes * profile.fotos_factor)
    + int(profile.solicitudes * profile.satisfaccion_factor)
    + 2922 * 4
    + 32
)

print(f'SCALE_MODE: {profile.name}')
print(f'Solicitudes: {profile.solicitudes:,}')
print(f'Eventos: {profile.eventos:,}')
print(f'Logs: {profile.logs:,}')
print(f'Shuffle partitions: {profile.shuffle_partitions}')
print(f'Filas Bronze esperadas: {expected_bronze_rows:,}')
print(f'Driver memory solicitada: {DRIVER_MEMORY}')

SCALE_MODE: full
Solicitudes: 50,000
Eventos: 10,800,000
Logs: 1,500,000
Shuffle partitions: 96
Filas Bronze esperadas: 12,429,220
Driver memory solicitada: 20g


After running the above cell, you should re-run the cell above (cell `74ce1d4c`) to see if the `ModuleNotFoundError` is resolved.

## 4. Ejecucion del pipeline

La ejecucion guarda evidencia por etapa en `evidence/execution_steps.json`. Si falla, el notebook no inventa resultados: captura el error, conserva `run_summary.json` y deja la etapa fallida registrada.

In [15]:
run_error = None
summary = None
start = time.time()

try:
    summary = run_pipeline(
        scale_mode=SCALE_MODE,
        root=PROJECT_ROOT,
        tmp_root=TMP_ROOT,
        warehouse_dir=WAREHOUSE_DIR,
        checkpoint_dir=CHECKPOINT_DIR,
        driver_memory=DRIVER_MEMORY,
    )
except Exception as exc:
    run_error = repr(exc)
    print('EJECUCION FALLIDA DE FORMA CONTROLADA:', run_error)
    summary_path = EVIDENCE_ROOT / 'run_summary.json'
    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
    else:
        summary = {'execution_status': 'failed', 'error_message': run_error}

print('Duracion notebook/pipeline segundos:', round(time.time() - start, 3))
print(json.dumps(summary, indent=2, ensure_ascii=False))

Duracion notebook/pipeline segundos: 2500.253
{
  "scale_mode": "full",
  "n_solicitudes_config": 50000,
  "n_eventos_config": 10800000,
  "n_logs_config": 1500000,
  "shuffle_partitions": 96,
  "started_at": "2026-07-03T16:07:36.421112+00:00",
  "execution_status": "success",
  "failed_step": null,
  "error_message": null,
  "ram_total_gb": 12.67,
  "ram_free_gb_start": 8.91,
  "disk_free_gb_start": 87.57,
  "spark_version": "4.0.3",
  "bronze_counts": {
    "solicitudes_raw": 50000,
    "eventos_raw": 10800000,
    "logs_raw": 1500000,
    "fotografias_metadata_raw": 40000,
    "clima_raw": 11688,
    "cuadrillas_raw": 32,
    "satisfaccion_raw": 27500
  },
  "bronze_total_rows": 12429220,
  "silver_counts": {
    "solicitudes_enriquecidas": 50000,
    "eventos_normalizados": 10800000,
    "logs_normalizados": 1500000,
    "fotografias_metadata": 40000,
    "satisfaccion": 27500,
    "clima": 11688,
    "cuadrillas": 32
  },
  "gold_counts": {
    "kpi_sla": 24,
    "backlog_operacio

## 5. Revision de evidencia generada

Usar inspecciones acotadas. Evitar `collect()`, `toPandas()` o visualizaciones completas sobre DataFrames grandes.

In [16]:
for file_name in ['run_summary.json', 'execution_steps.json', 'optimization_metrics.json']:
    path = EVIDENCE_ROOT / file_name
    print('\n---', file_name, '---')
    if path.exists():
        payload = json.loads(path.read_text(encoding='utf-8'))
        print(json.dumps(payload, indent=2, ensure_ascii=False)[:5000])
    else:
        print('No generado')



--- run_summary.json ---
{
  "scale_mode": "full",
  "n_solicitudes_config": 50000,
  "n_eventos_config": 10800000,
  "n_logs_config": 1500000,
  "shuffle_partitions": 96,
  "started_at": "2026-07-03T16:07:36.421112+00:00",
  "execution_status": "success",
  "failed_step": null,
  "error_message": null,
  "ram_total_gb": 12.67,
  "ram_free_gb_start": 8.91,
  "disk_free_gb_start": 87.57,
  "spark_version": "4.0.3",
  "bronze_counts": {
    "solicitudes_raw": 50000,
    "eventos_raw": 10800000,
    "logs_raw": 1500000,
    "fotografias_metadata_raw": 40000,
    "clima_raw": 11688,
    "cuadrillas_raw": 32,
    "satisfaccion_raw": 27500
  },
  "bronze_total_rows": 12429220,
  "silver_counts": {
    "solicitudes_enriquecidas": 50000,
    "eventos_normalizados": 10800000,
    "logs_normalizados": 1500000,
    "fotografias_metadata": 40000,
    "satisfaccion": 27500,
    "clima": 11688,
    "cuadrillas": 32
  },
  "gold_counts": {
    "kpi_sla": 24,
    "backlog_operacional": 8,
    "zonas_

## 6. KPIs Gold y dataset de priorizacion

Si la ejecucion completa llega a Gold, los datasets quedan en `data/gold`. El dataset de priorizacion evita data leakage: usa solo variables disponibles al ingreso o variables historicas/territoriales simuladas.

In [17]:
gold_path = DATA_ROOT / 'gold'
if gold_path.exists():
    print('Datasets Gold disponibles:')
    for path in sorted(gold_path.iterdir()):
        if path.is_dir():
            print('-', path.name)
else:
    print('Gold no disponible; revisar etapa fallida en execution_steps.json')

if summary:
    print('Conteos Gold reportados:', summary.get('gold_counts'))

Datasets Gold disponibles:
- backlog_operacional
- backlog_operacional_csv
- dataset_modelo_priorizacion
- desempeno_cuadrillas
- desempeno_cuadrillas_csv
- kpi_sla
- kpi_sla_csv
- satisfaccion_ciudadana
- satisfaccion_ciudadana_csv
- zonas_criticas
- zonas_criticas_csv
Conteos Gold reportados: {'kpi_sla': 24, 'backlog_operacional': 8, 'zonas_criticas': 4, 'satisfaccion_ciudadana': 12, 'desempeno_cuadrillas': 8, 'dataset_modelo_priorizacion': 50000}


## 7. Empaquetado final sin data pesada

Al finalizar, crear un ZIP liviano con `evidence/`, `docs/`, `README.md`, codigo y notebook. No se incluye `data/` para evitar mover Parquet masivo a Drive.

In [18]:
zip_path = Path('/content/fase2_repo_sin_data_full_evidence.zip') if IN_COLAB else PROJECT_ROOT / 'outputs' / 'fase2_repo_sin_data_full_evidence.zip'
zip_path.parent.mkdir(parents=True, exist_ok=True)

exclude_dirs = {'data', '.git', '__pycache__', '.pytest_cache', 'spark_tmp', 'spark_warehouse', 'spark_checkpoints'}
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(PROJECT_ROOT):
        dirs[:] = [d for d in dirs if d not in exclude_dirs and not d.endswith('_tmp')]
        for file in files:
            full_path = Path(root) / file
            if full_path == zip_path:
                continue
            rel_path = full_path.relative_to(PROJECT_ROOT)
            zf.write(full_path, rel_path)

evidence_zip = Path('/content/fase2_evidence_full.zip') if IN_COLAB else PROJECT_ROOT / 'outputs' / 'fase2_evidence_full.zip'
if EVIDENCE_ROOT.exists():
    shutil.make_archive(str(evidence_zip).replace('.zip', ''), 'zip', EVIDENCE_ROOT)

print('ZIP repo sin data:', zip_path)
print('ZIP evidence:', evidence_zip)
print('En Colab, copiar estos ZIP a Drive solo al final si se requiere respaldo.')

ZIP repo sin data: /content/fase2_repo_sin_data_full_evidence.zip
ZIP evidence: /content/fase2_evidence_full.zip
En Colab, copiar estos ZIP a Drive solo al final si se requiere respaldo.


## 8. Exportacion del notebook

En Colab usar `File > Download > Download .ipynb` y, si esta disponible, `File > Save and export notebook as > HTML`. Si se ejecuta localmente, puede usarse:

```bash
jupyter nbconvert --to html notebooks/fase2_pipeline_gestion_territorial.ipynb
```

## 9. Criterio de interpretacion

- Si `execution_status = success`, reportar conteos, tiempos y mejora desde `evidence/`.
- Si `execution_status = failed`, reportar `failed_step`, `error_message`, RAM/disco disponible y recomendacion Dataproc/Spark Serverless.
- No reducir `full` ni eliminar filas para declarar exito.